# 모델 학습 실행 (SPEC-02)
런타임이 새로 뜰 때마다 2 → 3 → 4 순서로 실행한다. 5는 결과 확인용(읽기만 함).
- 학습이 끊기면 같은 순서로 다시 실행하면 이어서 학습한다 (SPEC-02 5절).
- 준비 순서와 파일 위치의 근거: SPEC-02 "확정 후 보완 3"
- 입력 파일은 임시 위치 `MyDrive/ai-image-detector-tmp/` (SPEC-02 미해결 8, 공유 위치 정해지면 셀 2만 바꾼다)
- 커밋 전에 출력을 모두 지운다 (편집 → 모든 출력 지우기)
- Colab 보안 비밀에 GITHUB_TOKEN이 있어야 한다
- 평가(SPEC-03)는 셀 2·3 다음에 평가 실행 → 평가 결과 확인 순서로 실행한다. 학습 설정은 셀 2에서, 평가 설정(val/test)은 평가 실행 칸의 EVAL_CONFIG에서 바꾼다. val로 metrics.csv와 맞는지 확인한 뒤 test를 돌린다.

In [ ]:
# ===== 설정 (실험마다 이 셀만 바꾼다) =====
from pathlib import Path

CONFIG        = "src/model/configs/cifake_baseline.json"
DRIVE_DIR     = Path("/content/drive/MyDrive/ai-image-detector-tmp")  # 임시 위치 (SPEC-02 미해결 8)
ZIP_NAME      = "cifake.zip"
MANIFEST_NAME = "cifake_manifest.csv"
ZIP_SHA       = "cde8714fbe847e9b670be15b5b42a37b6b87f3f5481efc3032b0445351c54567"
MANIFEST_SHA  = "79e176eeb61bb024e6ffbc5163b5c84fa02f23daf74b9e39c2427086bcb3740c"
N_IMAGES      = 120000

ZIP_PATH  = Path("/content") / ZIP_NAME
DATA_ROOT = Path("/content/data")            # = config data_root
MANIFEST  = DATA_ROOT / MANIFEST_NAME        # = config manifest_path
REPO_URL  = "https://github.com/zeus1560/ai-image-detector.git"
REPO_DIR  = Path("/content/ai-image-detector")

In [ ]:
# ===== 준비 (런타임마다 실행) =====
import os, time, json, shutil, subprocess

def sh(cmd):
    # 토큰이 들어가지 않는 명령에만 사용
    return subprocess.run(cmd, shell=True, check=True, capture_output=True, text=True).stdout

def git_clone_private(repo_url, dest):
    """private 레포 clone. 토큰이 출력·예외 메시지에 나오지 않게 한다."""
    # 2. 위험한 위치(/content 자체, data, drive)를 지우지 않도록 막는다
    dest = Path(dest)
    assert dest.parent == Path("/content") and dest.name not in ("data", "drive"), f"clone 위치가 위험함: {dest}"
    # 3. https가 아니면 토큰이 들어가지 않아 원인 모를 실패가 나므로 미리 막는다
    assert repo_url.startswith("https://"), "repo_url은 https://로 시작해야 함"

    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    auth_url = repo_url.replace("https://", f"https://x-access-token:{token}@", 1)
    env = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}   # 인증 실패 시 입력 대기하지 않음
    if dest.exists():
        shutil.rmtree(dest)                            # pull 대신 새로 clone
    # check=False: CalledProcessError(명령 문자열 포함)가 생기지 않는다
    # 1. stderr는 받아 두고, 실패하면 토큰을 가린 뒤 보여준다
    r = subprocess.run(["git", "clone", "-q", auth_url, str(dest)],
                       stdout=subprocess.DEVNULL, stderr=subprocess.PIPE, text=True, env=env)
    del auth_url
    if r.returncode != 0:
        err = (r.stderr or "").replace(token, "***") if token else (r.stderr or "")
        del token
        shutil.rmtree(dest, ignore_errors=True)        # 반쯤 만들어진 폴더 제거
        raise RuntimeError(f"clone 실패, 종료 코드 {r.returncode}: {err.strip()}")
    del token
    # clone 직후 remote URL을 토큰 없는 주소로 교체
    r = subprocess.run(["git", "-C", str(dest), "remote", "set-url", "origin", repo_url],
                       stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    if r.returncode != 0:
        shutil.rmtree(dest, ignore_errors=True)        # 토큰이 남은 .git/config 제거
        raise RuntimeError(f"remote set-url 실패, 종료 코드 {r.returncode}")
    remote = sh(f"git -C {dest} remote get-url origin").strip()
    if "@" in remote:
        shutil.rmtree(dest, ignore_errors=True)
        raise RuntimeError("remote URL에 인증 정보가 남아 있어 폴더를 삭제함")
    return remote

# 0. GPU 확인
import torch
assert torch.cuda.is_available(), "GPU가 없습니다. 런타임 > 런타임 유형 변경에서 GPU를 선택하세요"
print("GPU:", torch.cuda.get_device_name(0))

# 1. Drive 연결 + zip 복사·확인
from google.colab import drive
drive.mount("/content/drive")
t = time.perf_counter()
shutil.copy(DRIVE_DIR / ZIP_NAME, ZIP_PATH)
assert sh(f"sha256sum {ZIP_PATH}").split()[0] == ZIP_SHA, f"{ZIP_NAME} sha256 불일치"
print(f"zip 복사·확인 {time.perf_counter()-t:.1f}s")

# 2. 압축 풀기 (DATA_ROOT를 새로 만듦)
if DATA_ROOT.exists():
    shutil.rmtree(DATA_ROOT)
t = time.perf_counter()
sh(f"unzip -q {ZIP_PATH} -d {DATA_ROOT}")
n_img = sum(1 for p in (DATA_ROOT / "images").rglob("*") if p.is_file())
assert n_img == N_IMAGES, f"이미지 수 {n_img} (기대 {N_IMAGES})"
print(f"unzip {n_img:,}개 {time.perf_counter()-t:.1f}s")

# 3. manifest 복사 (2번에서 DATA_ROOT를 지우므로 반드시 그 뒤에)
shutil.copy(DRIVE_DIR / MANIFEST_NAME, MANIFEST)
assert sh(f"sha256sum {MANIFEST}").split()[0] == MANIFEST_SHA, "manifest sha256 불일치"
print("manifest:", MANIFEST)

# 4. 레포 clone
remote = git_clone_private(REPO_URL, REPO_DIR)
print("레포:", remote, "@", sh(f"git -C {REPO_DIR} rev-parse --short HEAD").strip())

# 5. 설정 파일 경로가 위 준비 위치와 같은지 확인
cfg = json.loads((REPO_DIR / CONFIG).read_text(encoding="utf-8"))
assert Path(cfg["data_root"]) == DATA_ROOT, f"config data_root {cfg['data_root']} ≠ {DATA_ROOT}"
assert Path(cfg["manifest_path"]) == MANIFEST, f"config manifest_path {cfg['manifest_path']} ≠ {MANIFEST}"
print("config:", CONFIG, "run_name =", cfg["run_name"])
print("준비 완료")

In [ ]:
# ===== 학습 (끊기면 셀 2·3 후 이 셀을 다시 실행하면 이어서 학습) =====
!cd {REPO_DIR} && python -m src.model.train --config {CONFIG}

In [ ]:
# ===== 결과 확인: 실험 기록에 옮길 값 =====
run_dir = Path(cfg["output_dir"]) / cfg["run_name"]
for p in sorted(run_dir.iterdir()):
    print(f"{p.name:20s} {p.stat().st_size:>15,} B")
print()
print((run_dir / "metrics.csv").read_text(encoding="utf-8"))

In [ ]:
# ===== 평가 (SPEC-03): 준비 셀 실행 후. val을 먼저 돌려 metrics.csv와 맞는지 본 뒤 test =====
EVAL_CONFIG = "src/model/configs/cifake_baseline_eval_val.json"   # val → (확인 후) test

ecfg = json.loads((REPO_DIR / EVAL_CONFIG).read_text(encoding="utf-8"))
assert Path(ecfg["data_root"]) == DATA_ROOT, f"eval data_root {ecfg['data_root']} ≠ {DATA_ROOT}"
assert Path(ecfg["manifest_path"]) == MANIFEST, f"eval manifest_path {ecfg['manifest_path']} ≠ {MANIFEST}"
assert Path(ecfg["run_dir"]) == Path(cfg["output_dir"]) / cfg["run_name"], "평가 run_dir이 CONFIG의 학습 결과 폴더와 다름"
print("평가:", ecfg["eval_name"], "split =", ecfg["split"], "checkpoint =", ecfg["checkpoint"])
!cd {REPO_DIR} && python -m src.model.evaluate --config {EVAL_CONFIG}

In [ ]:
# ===== 평가 결과 확인: 실험 기록에 옮길 값 (완료 기준 B) =====
import pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score

eval_dir = Path(ecfg["run_dir"]) / "eval" / ecfg["eval_name"]
rec = json.loads((eval_dir / "eval_config.json").read_text(encoding="utf-8"))
summary = pd.read_csv(eval_dir / "summary.csv")
pred = pd.read_csv(eval_dir / "predictions.csv")
print(f"{ecfg['eval_name']}: {rec['checkpoint']}.pt epoch {rec['checkpoint_epoch']}, predictions {len(pred):,}행")
print(pred["label"].map({0: "real", 1: "fake"}).value_counts().to_string())
print(summary.to_string(index=False))

# B3: AP·ROC-AUC를 sklearn으로 다시 계산해 비교 (두 라벨이 다 있는 그룹만)
cond = summary[summary["group"] == "condition"]
for (s, c), g in pred.groupby(["source", "condition"]):
    if g["label"].nunique() < 2:
        continue
    row = cond[(cond["source"] == s) & (cond["condition"] == c)].iloc[0]
    print(f"({s}, {c}) sklearn 대비 AP 차이 {abs(average_precision_score(g['label'], g['score']) - row['ap']):.2e}, "
          f"AUC 차이 {abs(roc_auc_score(g['label'], g['score']) - row['auc']):.2e}")

# B1: val 평가면 metrics.csv의 같은 epoch 값과 비교 (정확도·오탐률·미탐률은 차이 0, AP·AUC는 1e-4 이내)
if ecfg["split"] == "val":
    m = pd.read_csv(Path(ecfg["run_dir"]) / "metrics.csv").set_index("epoch").loc[rec["checkpoint_epoch"]]
    row = cond[cond["condition"] == "original"].iloc[0]
    for k in ["accuracy", "false_positive_rate", "false_negative_rate", "ap", "auc"]:
        theirs = m[f"val_{k}"]
        print(f"{k:20s} 평가 {row[k]:.6f}  metrics.csv {theirs:.6f}  차이 {abs(row[k] - theirs):.2e}")